# Pydantic Utilities API Reference

Developer-facing statements defined in `libs/core/langchain_core/utils/pydantic.py`.

# `PYDANTIC_VERSION`

Parsed version of the installed Pydantic package.

```python
PYDANTIC_VERSION = version.parse(pydantic.__version__)
```

---

# `PYDANTIC_MAJOR_VERSION`

Major component of the installed Pydantic version.

```python
PYDANTIC_MAJOR_VERSION = PYDANTIC_VERSION.major
```

---

# `PYDANTIC_MINOR_VERSION`

Minor component of the installed Pydantic version.

```python
PYDANTIC_MINOR_VERSION = PYDANTIC_VERSION.minor
```

---

# `IS_PYDANTIC_V1`

Indicates whether this module is operating with Pydantic v1 as its primary version.

```python
IS_PYDANTIC_V1 = False
```

---

# `IS_PYDANTIC_V2`

Indicates whether this module is operating with Pydantic v2 as its primary version.

```python
IS_PYDANTIC_V2 = True
```

---

# `PydanticBaseModel`

Union of the Pydantic v2 and compatibility-v1 base model classes.

```python
PydanticBaseModel = BaseModel | BaseModelV1
```

---

# `TypeBaseModel`

Union of Pydantic v2 and compatibility-v1 model classes.

```python
TypeBaseModel = type[BaseModel] | type[BaseModelV1]
```

---

# `TBaseModel`

Type variable bounded to `PydanticBaseModel`.

```python
TBaseModel = TypeVar("TBaseModel", bound=PydanticBaseModel)
```

---

# `get_pydantic_major_version`

Returns the installed Pydantic major version.

```python
@deprecated("Use PYDANTIC_VERSION.major instead.")
get_pydantic_major_version(
) -> int # Installed Pydantic major version
```

---

# `is_pydantic_v1_subclass`

Checks whether a class derives from the Pydantic v1 compatibility `BaseModel`.

```python
is_pydantic_v1_subclass(
    cls: type, # Class to inspect
) -> bool # Whether cls is a Pydantic v1-compatible model class
```

The function delegates directly to `issubclass(cls, BaseModelV1)`.

---

# `is_pydantic_v2_subclass`

Checks whether a class derives from the Pydantic v2 `BaseModel`.

```python
is_pydantic_v2_subclass(
    cls: type, # Class to inspect
) -> bool # Whether cls is a Pydantic v2 model class
```

The function delegates directly to `issubclass(cls, BaseModel)`.

---

# `is_basemodel_subclass`

Checks whether an object is a Pydantic v2 or compatibility-v1 model class.

```python
is_basemodel_subclass(
    cls: type, # Candidate class
) -> bool # Whether cls derives from either supported BaseModel
```

Returns `False` when `cls` is not a class or is a `types.GenericAlias`. Otherwise, it checks both supported `BaseModel` classes.

---

# `is_basemodel_instance`

Checks whether an object is an instance of a Pydantic v2 or compatibility-v1 model.

```python
is_basemodel_instance(
    obj: Any, # Object to inspect
) -> bool # Whether obj is an instance of either supported BaseModel
```

---

# `pre_init`

Decorates a function so it runs as a Pydantic pre-initialization root validator.

`Callable` is imported only under `TYPE_CHECKING`; postponed annotations preserve the public type signature without requiring that name at runtime.

```python
pre_init(
    func: Callable[[Any, dict[str, Any]], Any], # Function receiving the model class and initialization values
) -> Callable[[Any, dict[str, Any]], Any] # Decorated pre-initialization validator
```

Before calling `func`, the wrapper maps populated aliases back to field names when the model permits population by field name. It also inserts defaults for non-required fields whose values are missing or `None`, invoking each field's default factory when present.

The decorator uses `@root_validator(pre=True)` to preserve validator ordering compatibility.

---

# `get_fields`

Returns the field mapping for a Pydantic v2 or compatibility-v1 model class or instance.

```python
@overload
get_fields(
    model: type[BaseModel], # Pydantic v2 model class
) -> dict[str, FieldInfoV2]
```

```python
@overload
get_fields(
    model: BaseModel, # Pydantic v2 model instance
) -> dict[str, FieldInfoV2]
```

```python
@overload
get_fields(
    model: type[BaseModelV1], # Pydantic v1-compatible model class
) -> dict[str, ModelField]
```

```python
@overload
get_fields(
    model: BaseModelV1, # Pydantic v1-compatible model instance
) -> dict[str, ModelField]
```

`ModelField` is imported only under `TYPE_CHECKING`.

Pydantic v2 inputs return `model_fields`; compatibility-v1 inputs return `__fields__`.

Raises `TypeError` when the supplied value is not a supported Pydantic model class or instance.

---

# `model_json_schema`

Returns the JSON Schema for a Pydantic v2 or compatibility-v1 model class.

```python
model_json_schema(
    model: TypeBaseModel, # Pydantic model class
) -> dict[str, Any] # Generated JSON Schema
```

Calls `model.model_json_schema()` for Pydantic v2 models and `model.schema()` for compatibility-v1 models.

Raises `TypeError` when `model` is not a supported Pydantic model class.

---

# `model_validate`

Validates an object against a Pydantic v2 or compatibility-v1 model class.

```python
model_validate(
    model: TypeBaseModel, # Pydantic model class used for validation
    obj: Any, # Object to validate
) -> PydanticBaseModel # Validated model instance
```

Calls `model.model_validate(obj)` for Pydantic v2 models and `model.parse_obj(obj)` for compatibility-v1 models.

Raises `TypeError` when `model` is not a supported Pydantic model class. Validation exceptions raised by Pydantic propagate unchanged.

---

# `create_model`

Creates a dynamic Pydantic v2 model.

The source recommends using `create_model_v2` instead. This function is not decorated as deprecated.

```python
create_model(
    model_name: str, # Name of the generated model
    module_name: str | None = None, # Module used to resolve forward references
    /,
    **field_definitions: Any, # Pydantic field definitions
) -> type[BaseModel] # Generated model class
```

When `field_definitions` contains `"__root__"`, that value is removed from the field definitions and forwarded as the `root` argument to `create_model_v2`. Other definitions are forwarded through `field_definitions`.

---

# `create_model_v2`

Creates a dynamic Pydantic v2 model from normal field definitions or a root type.

The source warns that this API should not be used outside LangChain packages and may change at any time.

```python
create_model_v2(
    model_name: str, # Name of the generated model
    *,
    module_name: str | None = None, # Module used to resolve forward references
    field_definitions: dict[str, Any] | None = None, # Pydantic field definitions
    root: Any | None = None, # Root type or a tuple containing the root type and default
) -> type[BaseModel] # Generated model class
```

When `root` is truthy, the function creates a `RootModel`. A two-item tuple is interpreted as the root type and its default value. The generated root model uses `model_name` as its JSON Schema title.

Raises `NotImplementedError` when a truthy `root` is supplied together with field definitions.

For normal models, arbitrary types are allowed, instances are frozen, and protected namespaces are disabled. Field names beginning with `_` or colliding with public `BaseModel` names are remapped to `private_<name>` while retaining the original name as the validation and serialization alias.

Raises `NotImplementedError` when a field requiring remapping is supplied directly as a Pydantic `FieldInfo` value.

Hashable model definitions are cached. When the cache arguments are unhashable, the model is created without the cache.

In [ ]:
# 1. Check the installed Pydantic version
from langchain_core.utils.pydantic import IS_PYDANTIC_V1, IS_PYDANTIC_V2, PYDANTIC_MAJOR_VERSION, PYDANTIC_MINOR_VERSION, PYDANTIC_VERSION # Import version constants


print("Full version:", PYDANTIC_VERSION) # Display the parsed Pydantic version
print("Major version:", PYDANTIC_MAJOR_VERSION) # Display the major version number
print("Minor version:", PYDANTIC_MINOR_VERSION) # Display the minor version number
print("Using Pydantic v1:", IS_PYDANTIC_V1) # Display whether primary v1 mode is active
print("Using Pydantic v2:", IS_PYDANTIC_V2) # Display whether primary v2 mode is active

In [ ]:
# 2. Detect Pydantic model classes and objects
from pydantic import BaseModel # Import the Pydantic v2 base model
from pydantic.v1 import BaseModel as BaseModelV1 # Import the Pydantic v1 compatibility model

from langchain_core.utils.pydantic import is_basemodel_instance, is_basemodel_subclass, is_pydantic_v1_subclass, is_pydantic_v2_subclass # Import model-detection helpers


class UserV2(BaseModel): # Define a Pydantic v2 model
    name: str # Store the user's name


class UserV1(BaseModelV1): # Define a Pydantic v1 compatibility model
    name: str # Store the user's name


user_v2 = UserV2(name="Saad") # Create a Pydantic v2 model instance
user_v1 = UserV1(name="Saad") # Create a Pydantic v1 model instance

print(is_pydantic_v2_subclass(UserV2)) # Display True for the Pydantic v2 model
print(is_pydantic_v1_subclass(UserV1)) # Display True for the Pydantic v1 model
print(is_basemodel_subclass(UserV2)) # Display True because UserV2 is a supported model class
print(is_basemodel_subclass(UserV1)) # Display True because UserV1 is a supported model class
print(is_basemodel_subclass(dict)) # Display False because dict is not a Pydantic model
print(is_basemodel_instance(user_v2)) # Display True for the Pydantic v2 instance
print(is_basemodel_instance(user_v1)) # Display True for the Pydantic v1 instance
print(is_basemodel_instance({"name": "Saad"})) # Display False for a normal dictionary

In [ ]:
# 3. Retrieve model fields
from pydantic import BaseModel, Field # Import model and field utilities

from langchain_core.utils.pydantic import get_fields # Import the cross-version field helper


class Employee(BaseModel): # Define an employee model
    employee_id: int = Field(description="Unique employee identifier") # Store the employee ID
    name: str = Field(description="Employee name") # Store the employee name
    department: str = "Engineering" # Store the department with a default value


employee = Employee(employee_id=101, name="Saad") # Create an employee instance

class_fields = get_fields(Employee) # Retrieve fields from the model class
instance_fields = get_fields(employee) # Retrieve fields from the model instance

print(class_fields.keys()) # Display the field names from the class
print(instance_fields.keys()) # Display the field names from the instance
print(class_fields["name"].description) # Display the description of the name field

In [ ]:
# 4. Generate JSON Schema
from langchain_core.utils.pydantic import model_json_schema # Import the cross-version schema generator


employee_schema = model_json_schema(Employee) # Generate JSON Schema for the Employee model

employee_schema # Display the generated schema in Jupyter

In [ ]:
# 5. Validate input data
from pydantic import ValidationError # Import Pydantic's validation exception

from langchain_core.utils.pydantic import model_validate # Import the cross-version validation helper


employee_data = { # Create raw employee data
    "employee_id": 102, # Provide the employee ID
    "name": "Aman", # Provide the employee name
    "department": "Analytics", # Provide the department
} # Finish creating the input dictionary

validated_employee = model_validate(Employee, employee_data) # Validate the dictionary against Employee

print(validated_employee) # Display the validated model
print(type(validated_employee).__name__) # Display the generated model class name

# Handle invalid input:
invalid_data = { # Create invalid employee data
    "employee_id": "not-an-integer", # Provide an invalid employee ID
    "name": "Aman", # Provide a valid employee name
} # Finish creating the invalid dictionary

try: # Begin validation error handling
    model_validate(Employee, invalid_data) # Attempt to validate the invalid data

except ValidationError as error: # Catch the propagated Pydantic validation error
    print(error) # Display the validation details


In [ ]:
# 6. Run logic before model initialization
from pydantic import BaseModel # Import the Pydantic base model

from langchain_core.utils.pydantic import pre_init # Import the pre-initialization decorator


class CourseRegistration(BaseModel): # Define a course-registration model
    student_name: str # Store the normalized student name
    course: str = "LangChain Fundamentals" # Provide a default course name
    active: bool = True # Provide a default registration status

    @pre_init # Run this method before normal model initialization
    def clean_input(cls, values: dict) -> dict: # Receive and transform the raw input values
        if "student_name" in values: # Check whether a student name was supplied
            values["student_name"] = values["student_name"].strip().title() # Remove spaces and normalize capitalization

        return values # Return the transformed initialization values


registration = CourseRegistration(student_name="  saad arfin  ") # Create a model using unclean input

print(registration.student_name) # Display Saad Arfin
print(registration.course) # Display the inserted default course
print(registration.active) # Display the inserted default status

In [ ]:
# 7. Create a dynamic model with create_model_v2()
from pydantic import Field # Import the Pydantic field helper

from langchain_core.utils.pydantic import create_model_v2 # Import LangChain's dynamic-model factory


Product = create_model_v2( # Create a dynamic Pydantic model
    "Product", # Set the generated class name
    field_definitions={ # Define the model fields
        "name": (str, Field(description="Product name")), # Create a required name field
        "price": (float, Field(gt=0, description="Product price")), # Create a positive price field
        "in_stock": (bool, True), # Create a Boolean field with a default value
    }, # Finish defining the fields
) # Finish creating the dynamic model

product = Product(name="Laptop", price=75000) # Create an instance of the generated model

print(product) # Display the validated product
print(product.model_dump()) # Convert the generated model to a dictionary
print(Product.model_json_schema()) # Display its generated JSON Schema


In [ ]:
# 8. Create a dynamic root model
from langchain_core.utils.pydantic import create_model_v2 # Import the dynamic-model factory


SkillList = create_model_v2( # Create a root model containing a list
    "SkillList", # Set the generated model name
    root=list[str], # Define the root value as a list of strings
) # Finish creating the root model

skills = SkillList(["Python", "SQL", "LangChain"]) # Validate a list through the root model

print(skills.root) # Display the validated root list
print(SkillList.model_json_schema()) # Display the root model's JSON Schema